In [5]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator



data_dir = "Garbage classification dataset"



In [6]:
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_input_resnet
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from sklearn.metrics import confusion_matrix


In [7]:
train_gen1 = ImageDataGenerator(
    preprocessing_function=preprocess_input_resnet,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_gen1 = ImageDataGenerator(
    preprocessing_function=preprocess_input_resnet,
    validation_split=0.2
)

train_data1 = train_gen1.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

test_data1 = val_gen1.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False

)


Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.


In [8]:
train_labels = train_data1.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_array))
print("Class Weights:", class_weights)


Class Weights: {0: np.float64(1.04437564499484), 1: np.float64(0.8412302576891105), 2: np.float64(1.0284552845528456), 3: np.float64(0.7086834733893558), 4: np.float64(0.8739205526770294), 5: np.float64(3.066666666666667)}


In [9]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

resnet = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
resnet.trainable = False  # freeze pretrained layers

# Build Model
model = Sequential([
    resnet,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(train_data1.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_data1,
    validation_data=test_data1,
    epochs=20,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stop]
)

print("Training completed!")


Epoch 1/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5717 - loss: 1.3111

64/64 ━━━━━━━━━━━━━━━━━━━━ 270s 4s/step - accuracy: 0.6917 - loss: 0.9305 - val_accuracy: 0.7694 - val_loss: 0.6334
Epoch 2/20
 1/64 ━━━━━━━━━━━━━━━━━━━━ 6:37 6s/step - accuracy: 0.8438 - loss: 0.3512

AbortedError: Graph execution error:

Detected at node StatefulPartitionedCall/sequential_1/resnet50_1/conv4_block1_2_conv_1/BiasAdd defined at (most recent call last):
<stack traces unavailable>
Operation received an exception:Status: 1, message: could not create a memory object, in file tensorflow/core/kernels/mkl/mkl_conv_ops.cc:1112
	 [[{{node StatefulPartitionedCall/sequential_1/resnet50_1/conv4_block1_2_conv_1/BiasAdd}}]] [Op:__inference_multi_step_on_iterator_11720]

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(test_data1)
y_pred = np.argmax(pred, axis=1)
y_true = test_data1.classes

#print(classification_report(y_true, y_pred, target_names=list(test_data.class_indices.keys())))
cm = confusion_matrix(y_true, y_pred)
cm
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

16/16 ━━━━━━━━━━━━━━━━━━━━ 60s 3s/step
Accuracy: 0.8349900596421471


Fine tuning

In [ ]:
resnet.trainable = True

# keep early layers frozen (good for transfer learning)
for layer in resnet.layers[:-120]:
    layer.trainable = False


NameError: name 'resnet' is not defined

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history_finetune = model.fit(
    train_data1,
    validation_data=test_data1,
    epochs=10,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stop]
)


Epoch 1/10


: 

In [ ]:
test_data1.reset()

pred1 = model.predict(test_data1)
y_pred1 = np.argmax(pred1, axis=1)
accuracy1 = np.mean(y_pred1 == test_data1.classes)

print("New Accuracy after fine-tuning:", accuracy1)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Reset generator
#test_data1.reset()

# Predictions
pred1 = model.predict(test_data1)
y_pred1 = np.argmax(pred1, axis=1)
y_true1 = test_data1.classes

# Confusion matrix
cm_resnet= confusion_matrix(y_true1, y_pred1)

# Class names
class_names = list(test_data1.class_indices.keys())

# Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Fine-Tuned Model)")
plt.show()


ModuleNotFoundError: No module named 'seaborn'